# GTSRB — Traffic Sign Recognition
Deep Learning Course Project — Custom CNN vs. ResNet-50 (Transfer Learning).

In [ ]:
# Section 1: Imports and global configuration
import time, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms
from torchvision.datasets import GTSRB
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility: fixed seed across python, numpy, and torch (CPU + CUDA)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Performance: cuDNN autotuner picks the fastest conv algorithm for fixed input shape
torch.backends.cudnn.benchmark = True

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE    = 64    # 64x64 keeps digit detail (30 vs 80) while staying fast to train
NUM_CLASSES = 43    # GTSRB has 43 traffic-sign categories
USE_AMP     = torch.cuda.is_available()   # mixed precision only on GPU

print(f'Device      : {DEVICE}')
print(f'Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'Classes     : {NUM_CLASSES}')
print(f'Mixed prec. : {USE_AMP}')

In [ ]:
# Section 2: Dataset download, class names, and per-class exploration

# First run downloads ~300 MB; subsequent runs use the local cache
trainval_raw = GTSRB(root='./data', split='train', download=True)
test_raw     = GTSRB(root='./data', split='test',  download=True)

# Short labels used in plots and the classification report
CLASS_NAMES = [
    'Speed limit (20km/h)',        'Speed limit (30km/h)',
    'Speed limit (50km/h)',        'Speed limit (60km/h)',
    'Speed limit (70km/h)',        'Speed limit (80km/h)',
    'End of speed limit (80km/h)', 'Speed limit (100km/h)',
    'Speed limit (120km/h)',       'No passing',
    'No passing veh > 3.5t',       'Right-of-way at intersection',
    'Priority road',               'Yield',
    'Stop',                        'No vehicles',
    'Veh > 3.5t prohibited',       'No entry',
    'General caution',             'Dangerous curve left',
    'Dangerous curve right',       'Double curve',
    'Bumpy road',                  'Slippery road',
    'Road narrows right',          'Road work',
    'Traffic signals',             'Pedestrians',
    'Children crossing',           'Bicycles crossing',
    'Beware ice/snow',             'Wild animals crossing',
    'End speed+passing limits',    'Turn right ahead',
    'Turn left ahead',             'Ahead only',
    'Go straight or right',        'Go straight or left',
    'Keep right',                  'Keep left',
    'Roundabout mandatory',        'End of no passing',
    'End no passing veh > 3.5t',
]

# Plain-language meaning per class, overlaid on prediction images later
SIGN_DESCRIPTIONS = {
    0:'This sign sets a speed limit of 20 km/h.',
    1:'This sign sets a speed limit of 30 km/h.',
    2:'This sign sets a speed limit of 50 km/h.',
    3:'This sign sets a speed limit of 60 km/h.',
    4:'This sign sets a speed limit of 70 km/h.',
    5:'This sign sets a speed limit of 80 km/h.',
    6:'This sign ends the 80 km/h speed limit.',
    7:'This sign sets a speed limit of 100 km/h.',
    8:'This sign sets a speed limit of 120 km/h.',
    9:'This sign prohibits overtaking.',
    10:'This sign bans overtaking by vehicles over 3.5 t.',
    11:'This sign grants right-of-way at the next intersection.',
    12:'This sign marks the road as a priority road.',
    13:'This sign means yield to oncoming traffic.',
    14:'This sign means stop completely before proceeding.',
    15:'This sign means all vehicles are prohibited.',
    16:'This sign bans vehicles heavier than 3.5 t.',
    17:'This sign means do not enter.',
    18:'This sign warns of a general hazard ahead.',
    19:'This sign warns of a dangerous left curve ahead.',
    20:'This sign warns of a dangerous right curve ahead.',
    21:'This sign warns of a double curve ahead.',
    22:'This sign warns of a bumpy road surface ahead.',
    23:'This sign warns of a slippery road ahead.',
    24:'This sign warns that the road narrows on the right.',
    25:'This sign warns of road work ahead.',
    26:'This sign warns of traffic signals ahead.',
    27:'This sign warns of pedestrians crossing.',
    28:'This sign warns of children crossing.',
    29:'This sign warns of bicycles crossing.',
    30:'This sign warns of ice or snow on the road.',
    31:'This sign warns of wild animals on the road.',
    32:'This sign ends all speed and passing restrictions.',
    33:'This sign means you must turn right ahead.',
    34:'This sign means you must turn left ahead.',
    35:'This sign means go straight, no turning.',
    36:'This sign means go straight or turn right.',
    37:'This sign means go straight or turn left.',
    38:'This sign means keep to the right side.',
    39:'This sign means keep to the left side.',
    40:'This sign indicates a roundabout ahead.',
    41:'This sign ends the no-passing zone.',
    42:'This sign ends no-passing for vehicles over 3.5 t.',
}

# Build a class_id -> indices map once, so we can sample per class without rescanning.
# We read labels only (img discarded) to keep this fast.
all_labels = np.array([lbl for _, lbl in trainval_raw])
class_to_idx = {c: np.where(all_labels == c)[0].tolist() for c in range(NUM_CLASSES)}
counts = np.array([len(class_to_idx[c]) for c in range(NUM_CLASSES)])

print(f'Train+Val : {len(trainval_raw):,} images')
print(f'Test      : {len(test_raw):,} images')
print(f'Imbalance : most common {counts.max()} vs rarest {counts.min()} samples '
      f'(ratio {counts.max()/counts.min():.1f}x)')

# Class-distribution bar chart (motivates stratified split + class weighting later)
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(NUM_CLASSES), counts, color='steelblue', edgecolor='white', lw=0.3)
ax.axhline(counts.mean(), color='red', linestyle='--', label=f'Mean = {counts.mean():.0f}')
ax.set_title('GTSRB - Samples per Class'); ax.set_xlabel('Class ID'); ax.legend()
plt.tight_layout(); plt.savefig('class_distribution.png', dpi=110); plt.show()

# One random sample per class
fig, axes = plt.subplots(6, 8, figsize=(18, 14))
fig.suptitle('One Sample per Class (43 classes)', fontsize=12, fontweight='bold')
for cid in range(NUM_CLASSES):
    ax = axes[cid // 8][cid % 8]
    img, _ = trainval_raw[random.choice(class_to_idx[cid])]
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'[{cid}] {CLASS_NAMES[cid][:14]}', fontsize=5.5)
for i in range(NUM_CLASSES, 48):
    axes[i // 8][i % 8].axis('off')
plt.tight_layout(); plt.savefig('all_classes.png', dpi=110); plt.show()

In [ ]:
# Section 3: Preprocessing and data augmentation

# ImageNet channel statistics. Used for BOTH models so the pre-trained ResNet
# backbone sees the input distribution it was trained on.
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

# Training pipeline: random affine + colour jitter simulate camera angle and lighting.
# Horizontal flip is intentionally excluded because it changes a sign's meaning
# (a right-arrow becomes a left-arrow).
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=12, translate=(0.1, 0.1),
                            scale=(0.9, 1.1), shear=8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Evaluation pipeline: deterministic, no augmentation.
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Visual sanity check: one original image vs five augmented versions
src_id  = random.choice(class_to_idx[random.randint(0, 42)])
src_img, src_lbl = trainval_raw[src_id]
mean_t, std_t = torch.tensor(MEAN).view(3,1,1), torch.tensor(STD).view(3,1,1)

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
fig.suptitle(f'Augmentation Demo - {CLASS_NAMES[src_lbl]}', fontsize=11)
orig = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor()])(src_img)
axes[0].imshow(orig.permute(1,2,0)); axes[0].set_title('Original'); axes[0].axis('off')
for i in range(1, 6):
    aug  = train_tf(src_img)
    disp = (aug * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()
    axes[i].imshow(disp); axes[i].set_title(f'Aug {i}'); axes[i].axis('off')
plt.tight_layout(); plt.savefig('augmentation_demo.png', dpi=110); plt.show()

In [ ]:
# Section 4: Stratified split and DataLoaders

class IndexedDataset(Dataset):
    """Wrap a torchvision dataset with (optional) index subset + a transform.
    Centralises both the 'subset' and 'transform' wrappers used before.
    """
    def __init__(self, base, transform, indices=None):
        self.base, self.tf = base, transform
        self.indices = list(indices) if indices is not None else None
    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.base)
    def __getitem__(self, i):
        j = self.indices[i] if self.indices is not None else i
        img, lbl = self.base[j]
        return self.tf(img), lbl

# Stratified 85/15 split: keeps the per-class ratio in train and val, which matters
# because GTSRB is heavily imbalanced (rarest class has only ~200 samples).
train_idx, val_idx = train_test_split(
    np.arange(len(trainval_raw)),
    test_size=0.15,
    stratify=all_labels,
    random_state=SEED,
)

train_ds = IndexedDataset(trainval_raw, train_tf, train_idx)   # augmented
val_ds   = IndexedDataset(trainval_raw, eval_tf,  val_idx)     # clean
test_ds  = IndexedDataset(test_raw,     eval_tf)               # clean

loader_kw    = dict(num_workers=2, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, **loader_kw)

# Class weights for the loss: inverse-frequency, normalised to mean = 1.
# This compensates for class imbalance without changing the sampler.
train_labels = all_labels[train_idx]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
class_weights = (class_counts.mean() / np.clip(class_counts, 1, None))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print(f'Train   : {len(train_ds):,} images   ({len(train_loader)} batches)')
print(f'Val     : {len(val_ds):,} images   ({len(val_loader)} batches)')
print(f'Test    : {len(test_ds):,} images   ({len(test_loader)} batches)')
print(f'Class weights range: [{class_weights.min():.2f}, {class_weights.max():.2f}]')

In [ ]:
# Section 5: Model 1 - Custom CNN, trained from scratch
#
# Architecture (input 3x64x64):
#   Block 1: Conv x2 + BN + ReLU + MaxPool + Dropout  -> 32 x 32 x 32
#   Block 2: Conv x2 + BN + ReLU + MaxPool + Dropout  -> 64 x 16 x 16
#   Block 3: Conv x2 + BN + ReLU + MaxPool + Dropout  -> 128 x 8 x 8
#   Global Average Pool                               -> 128
#   FC(128->256) + BN + ReLU + Dropout + FC(256->43)  -> 43 logits

class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        def conv_block(in_ch, out_ch):
            # Two 3x3 convs per block: richer features before each downsample.
            return nn.Sequential(
                nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2),
                nn.Dropout(p=0.25),
            )

        self.block1 = conv_block(3,   32)    # edges, colour gradients
        self.block2 = conv_block(32,  64)    # shapes, corners
        self.block3 = conv_block(64,  128)   # sign-level patterns, digits

        # GAP collapses each feature map to a single value: fewer params than Flatten,
        # also acts as a structural regulariser.
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes),   # raw logits; softmax is folded into the loss
        )

        # Kaiming init keeps activation variance stable through ReLU layers.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        return self.classifier(x)


cnn_model = TrafficSignCNN(NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'Custom CNN - parameters: {total_params:,}')

# Shape sanity check
with torch.no_grad():
    dummy_out = cnn_model(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
print(f'Output shape: {tuple(dummy_out.shape)}  OK')

In [ ]:
# Section 6: Model 2 - ResNet-50 with transfer learning
#
# Strategy:
#   1. Load ResNet-50 pre-trained on ImageNet (1.28M images, 1000 classes).
#   2. Freeze every layer to preserve the learned visual features.
#   3. Replace the final FC layer: 2048 -> 1000  becomes  2048 -> 43.
#   4. Train only the new head (~88K parameters out of ~25.5M).

resnet_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Freeze backbone: gradients are neither computed nor applied for these layers.
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the classification head
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)
resnet_model    = resnet_model.to(DEVICE)

total     = sum(p.numel() for p in resnet_model.parameters())
trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'ResNet-50 - total params: {total:,}   trainable: {trainable:,}')
print(f'Only {trainable/total*100:.2f}% of weights are updated during training')

with torch.no_grad():
    dummy_out = resnet_model(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
print(f'Output shape: {tuple(dummy_out.shape)}  OK')

In [ ]:
# Section 7: Shared training loop (CNN + ResNet)
#
# Professional touches included here:
#   - Class-weighted CrossEntropy + label smoothing (handles imbalance)
#   - Adam with weight decay + cosine LR schedule
#   - Gradient clipping (max_norm=1.0)
#   - Mixed-precision training on GPU (autocast + GradScaler)
#   - Early stopping on validation accuracy with best-checkpoint saving

def train_model(model, model_name, epochs, lr, checkpoint, patience=5):
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = GradScaler(enabled=USE_AMP)

    history = {'tr_loss': [], 'tr_acc': [], 'va_loss': [], 'va_acc': []}
    best_acc, patience_count = 0.0, 0

    print(f'\nTraining: {model_name}   lr={lr}   max_epochs={epochs}')
    header = f'{"Ep":>4}  {"Tr Loss":>8}  {"Tr Acc":>7}  {"Va Loss":>8}  {"Va Acc":>7}  {"Time":>5}'
    print(header)

    for ep in range(1, epochs + 1):
        t0 = time.time()

        # Training
        model.train()
        tr_loss = tr_correct = tr_n = 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=USE_AMP):
                logits = model(x)
                loss   = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            tr_loss    += loss.item() * y.size(0)
            tr_correct += (logits.argmax(1) == y).sum().item()
            tr_n       += y.size(0)

        # Validation
        model.eval()
        va_loss = va_correct = va_n = 0
        with torch.no_grad(), autocast(enabled=USE_AMP):
            for x, y in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                logits = model(x)
                va_loss    += criterion(logits, y).item() * y.size(0)
                va_correct += (logits.argmax(1) == y).sum().item()
                va_n       += y.size(0)

        tr_l, tr_a = tr_loss/tr_n, tr_correct/tr_n
        va_l, va_a = va_loss/va_n, va_correct/va_n
        history['tr_loss'].append(tr_l); history['tr_acc'].append(tr_a)
        history['va_loss'].append(va_l); history['va_acc'].append(va_a)
        scheduler.step()

        # Best-checkpoint + early stopping
        flag = ''
        if va_a > best_acc:
            best_acc, patience_count = va_a, 0
            torch.save(model.state_dict(), checkpoint)
            flag = '  <- best'
        else:
            patience_count += 1

        print(f'{ep:>4}  {tr_l:>8.4f}  {tr_a*100:>6.2f}%  '
              f'{va_l:>8.4f}  {va_a*100:>6.2f}%  {time.time()-t0:>4.1f}s{flag}')

        if patience_count >= patience:
            print(f'  Early stopping at epoch {ep}.')
            break

    print(f'  Best val accuracy: {best_acc*100:.2f}%')
    return history

In [ ]:
# Section 8: Run training for both models

# Custom CNN from scratch
history_cnn = train_model(
    model=cnn_model, model_name='Custom CNN',
    epochs=25, lr=1e-3, checkpoint='best_cnn.pt',
)

# ResNet-50: head only (backbone is frozen, so a normal LR is safe)
history_resnet = train_model(
    model=resnet_model, model_name='ResNet-50 (Transfer Learning)',
    epochs=15, lr=1e-3, checkpoint='best_resnet.pt',
)

In [ ]:
# Section 8b (optional): Fine-tune ResNet-50 layer4 with a small LR
# Standard transfer-learning practice: after the head is well-trained, unfreeze
# the deepest residual block and train it with a small LR. This typically adds
# +0.3-0.8% test accuracy at low risk of forgetting ImageNet features.

for param in resnet_model.layer4.parameters():
    param.requires_grad = True

history_resnet_ft = train_model(
    model=resnet_model, model_name='ResNet-50 (Fine-tune layer4)',
    epochs=5, lr=1e-4, checkpoint='best_resnet.pt', patience=3,
)

In [ ]:
# Section 9: Training curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves: Custom CNN vs ResNet-50', fontsize=13, fontweight='bold')

for ax, key, ylabel in [
    (axes[0], 'loss', 'Loss'),
    (axes[1], 'acc',  'Accuracy (%)'),
]:
    cnn_tr = history_cnn[f'tr_{key}'];    cnn_va = history_cnn[f'va_{key}']
    res_tr = history_resnet[f'tr_{key}']; res_va = history_resnet[f'va_{key}']
    if key == 'acc':
        cnn_tr = [v*100 for v in cnn_tr]; cnn_va = [v*100 for v in cnn_va]
        res_tr = [v*100 for v in res_tr]; res_va = [v*100 for v in res_va]

    ax.plot(cnn_tr,  '--', color='steelblue',  lw=1.5, label='CNN - Train')
    ax.plot(cnn_va,  '-',  color='steelblue',  lw=2.0, label='CNN - Val')
    ax.plot(res_tr,  '--', color='darkorange', lw=1.5, label='ResNet - Train')
    ax.plot(res_va,  '-',  color='darkorange', lw=2.0, label='ResNet - Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig('training_curves.png', dpi=120); plt.show()

In [ ]:
# Section 10: Test-set evaluation

@torch.no_grad()
def evaluate(model, checkpoint):
    """Load best checkpoint, run on the full test set, return predictions + metrics."""
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()

    all_preds, all_labels_, all_probs = [], [], []
    for x, y in test_loader:
        x = x.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(x)
        probs = F.softmax(logits.float(), dim=1).cpu().numpy()
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels_.append(y.numpy())
        all_probs.append(probs)

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels_)
    probs  = np.concatenate(all_probs)

    top1 = (y_pred == y_true).mean()
    # Vectorised top-5: check whether the true label sits inside the top-5 indices.
    top5_idx = np.argpartition(-probs, 5, axis=1)[:, :5]
    top5 = np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))])

    return {'y_pred': y_pred, 'y_true': y_true, 'probs': probs,
            'top1': top1, 'top5': top5}


results_cnn    = evaluate(cnn_model,    'best_cnn.pt')
results_resnet = evaluate(resnet_model, 'best_resnet.pt')

# Comparison table
print('='*52)
print(f'{"Metric":<22}  {"Custom CNN":>12}  {"ResNet-50":>12}')
print('='*52)
print(f'{"Top-1 Accuracy":<22}  {results_cnn["top1"]*100:>11.2f}%  {results_resnet["top1"]*100:>11.2f}%')
print(f'{"Top-5 Accuracy":<22}  {results_cnn["top5"]*100:>11.2f}%  {results_resnet["top5"]*100:>11.2f}%')
print(f'{"Human Benchmark":<22}  {"98.84%":>12}  {"98.84%":>12}')
print('='*52)

# Normalised confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Normalised Confusion Matrix (row = true class)', fontsize=12)
for ax, res, name in [
    (axes[0], results_cnn,    f'Custom CNN  ({results_cnn["top1"]*100:.2f}%)'),
    (axes[1], results_resnet, f'ResNet-50   ({results_resnet["top1"]*100:.2f}%)'),
]:
    cm = confusion_matrix(res['y_true'], res['y_pred']).astype(float)
    cm /= cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=120); plt.show()

# Per-class report for the best model
best = results_resnet if results_resnet['top1'] > results_cnn['top1'] else results_cnn
print('\nPer-class report (best model):')
print(classification_report(best['y_true'], best['y_pred'],
                             target_names=[n[:28] for n in CLASS_NAMES], digits=3))

# Most confused class pairs - a useful diagnostic that reveals visually similar signs
cm_best = confusion_matrix(best['y_true'], best['y_pred'])
off_diag = cm_best.copy(); np.fill_diagonal(off_diag, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off_diag.ravel())[::-1][:5], off_diag.shape))[0]
print('\nTop-5 most confused pairs (true -> predicted, count):')
for t, p in pairs:
    print(f'  {CLASS_NAMES[t][:28]:<30} -> {CLASS_NAMES[p][:28]:<30}  {cm_best[t, p]}')

In [ ]:
# Section 11: Prediction visualisation
#
# For each of 8 random test images we show:
#   - the sign photograph,
#   - a text overlay describing the predicted sign in plain language,
#   - title with confidence and a green check / red cross indicator.
# Both models are evaluated on the SAME 8 images.

cnn_model.load_state_dict(torch.load('best_cnn.pt',       map_location=DEVICE))
resnet_model.load_state_dict(torch.load('best_resnet.pt', map_location=DEVICE))
cnn_model.eval(); resnet_model.eval()

# Fix seed so both runs pick the same images
random.seed(SEED)
sample_ids = random.sample(range(len(test_raw)), 8)


def show_predictions(model, model_name, title_color):
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    fig.suptitle(f'{model_name} - Predictions on 8 Test Images',
                 fontsize=14, fontweight='bold', color=title_color, y=1.01)

    for ax, idx in zip(axes.flat, sample_ids):
        img_pil, true_label = test_raw[idx]

        x = eval_tf(img_pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(x), dim=1)[0].cpu().numpy()

        pred    = int(probs.argmax())
        conf    = float(probs[pred]) * 100
        correct = (pred == true_label)

        ax.imshow(img_pil); ax.axis('off')

        # Description overlay (plain-language meaning of the predicted sign)
        ax.text(
            0.5, 0.03, SIGN_DESCRIPTIONS[pred],
            transform=ax.transAxes,
            fontsize=8, color='white', ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.72),
        )

        color = '#1a7a3e' if correct else '#c0392b'
        title = (f'OK  {conf:.1f}% confidence' if correct
                 else f'X   {conf:.1f}% confidence\nTrue: {CLASS_NAMES[true_label]}')
        ax.set_title(title, fontsize=9, color=color, fontweight='bold', pad=4)

    plt.tight_layout()
    plt.savefig(f'predictions_{model_name.replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()


show_predictions(cnn_model,    'Custom CNN',  'steelblue')
show_predictions(resnet_model, 'ResNet-50',   'darkorange')

In [ ]:
# Section 12: Final benchmark comparison

benchmarks = {
    'HOG + SVM (Classical ML)':               95.71,
    'Human Performance (Stallkamp 2012)':      98.84,
    'Model 1 - Custom CNN':                    results_cnn['top1']    * 100,
    'Model 2 - ResNet-50 (Transfer Learning)': results_resnet['top1'] * 100,
}

names  = list(benchmarks.keys())
values = list(benchmarks.values())
colors = ['#aab7b8', '#aab7b8', 'steelblue', 'darkorange']

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(names, values, color=colors, edgecolor='black', lw=0.6, height=0.5)
ax.set_xlim(93, 101)
ax.set_xlabel('Test Top-1 Accuracy (%)', fontsize=11)
ax.set_title('GTSRB - Model Comparison', fontsize=13, fontweight='bold')
ax.axvline(98.84, color='#555', linestyle='--', lw=1.2, label='Human: 98.84%')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.08, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=9)
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('benchmark.png', dpi=120); plt.show()

print('\nProject complete. Saved files:')
for f in ['class_distribution.png','all_classes.png','augmentation_demo.png',
          'training_curves.png','confusion_matrices.png','benchmark.png',
          'predictions_Custom_CNN.png','predictions_ResNet-50.png',
          'best_cnn.pt','best_resnet.pt']:
    print(f'  {f}')